In [6]:
# ============================================================
# OLIST E-COMMERCE DATA PIPELINE — Google Colab
# Member 1 — Data Processing & Dataset Module
# Output: final_cleaned_data.csv
# ============================================================

# ── CELL 1 · Install / imports ────────────────────────────────
!pip install -q tabulate

import zipfile, os, warnings
import pandas as pd
import numpy as np
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

print("✅ Libraries ready")


# ── CELL 2 · Upload & extract zip ─────────────────────────────
from google.colab import files
print("Upload your archive.zip ↓")
uploaded = files.upload()
ZIP_PATH = list(uploaded.keys())[0]

EXTRACT_DIR = "/content/olist_raw"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

csv_files = [f for f in os.listdir(EXTRACT_DIR) if f.endswith(".csv")]
print(f"\n📂 Extracted {len(csv_files)} files:")
for f in sorted(csv_files):
    size_mb = os.path.getsize(os.path.join(EXTRACT_DIR, f)) / 1_000_000
    print(f"   {f:55s}  {size_mb:6.1f} MB")


# ── CELL 3 · Load raw CSVs ────────────────────────────────────
def load(name):
    return pd.read_csv(os.path.join(EXTRACT_DIR, name))

orders    = load("olist_orders_dataset.csv")
customers = load("olist_customers_dataset.csv")
items     = load("olist_order_items_dataset.csv")
payments  = load("olist_order_payments_dataset.csv")
reviews   = load("olist_order_reviews_dataset.csv")
products  = load("olist_products_dataset.csv")
sellers   = load("olist_sellers_dataset.csv")
cat_xlat  = load("product_category_name_translation.csv")

print("✅ All CSVs loaded")
for name, df in [("orders", orders), ("customers", customers),
                 ("items", items), ("payments", payments),
                 ("reviews", reviews), ("products", products),
                 ("sellers", sellers)]:
    print(f"   {name:12s}  {df.shape[0]:>7,} rows × {df.shape[1]} cols")


# ── CELL 4 · Clean each table ─────────────────────────────────

# ── 4a. ORDERS ───────────────────────────────────────────────
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")   # ✅ FIX 1: proper datetime

orders.dropna(subset=["order_purchase_timestamp"], inplace=True)

valid_statuses = {"delivered", "shipped", "processing",
                  "canceled", "invoiced", "approved"}
orders = orders[orders["order_status"].isin(valid_statuses)].copy()

orders["days_to_delivery"] = (
    orders["order_delivered_customer_date"] -
    orders["order_purchase_timestamp"]
).dt.days
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_estimated_delivery_date"]
).dt.days   # negative = arrived early

print(f"✅ Orders cleaned  →  {orders.shape[0]:,} rows")


# ── 4b. CUSTOMERS ────────────────────────────────────────────
customers["customer_city"]  = customers["customer_city"].str.strip().str.title()
customers["customer_state"] = customers["customer_state"].str.strip().str.upper()
customers.dropna(subset=["customer_id"], inplace=True)
customers.drop_duplicates(subset=["customer_id"], inplace=True)
print(f"✅ Customers cleaned  →  {customers.shape[0]:,} rows")


# ── 4c. ORDER ITEMS ──────────────────────────────────────────
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")
items["price"]         = pd.to_numeric(items["price"],         errors="coerce")
items["freight_value"] = pd.to_numeric(items["freight_value"], errors="coerce")
items.dropna(subset=["order_id", "product_id", "price"], inplace=True)
items["total_item_value"] = items["price"] + items["freight_value"]
print(f"✅ Items cleaned  →  {items.shape[0]:,} rows")


# ── 4d. PAYMENTS ─────────────────────────────────────────────
payments["payment_value"] = pd.to_numeric(payments["payment_value"], errors="coerce")
payments.dropna(subset=["order_id", "payment_value"], inplace=True)

payments_agg = (
    payments.groupby("order_id")
    .agg(
        payment_types    = ("payment_type",        lambda x: "|".join(x.unique())),
        payment_value    = ("payment_value",        "sum"),
        max_installments = ("payment_installments", "max"),
    )
    .reset_index()
)
print(f"✅ Payments cleaned  →  {payments_agg.shape[0]:,} rows")


# ── 4e. REVIEWS ──────────────────────────────────────────────
reviews["review_score"] = pd.to_numeric(reviews["review_score"], errors="coerce")
reviews = reviews[reviews["review_score"].between(1, 5)].copy()
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c], errors="coerce")
reviews.drop_duplicates(subset=["order_id"], keep="last", inplace=True)
reviews_slim = reviews[["order_id", "review_score", "review_comment_message"]].copy()

# ✅ FIX 2: fill missing review comments so teammates don't hit NaN errors
reviews_slim["review_comment_message"] = reviews_slim["review_comment_message"].fillna("")

print(f"✅ Reviews cleaned  →  {reviews_slim.shape[0]:,} rows")


# ── 4f. PRODUCTS ─────────────────────────────────────────────
products = products.merge(cat_xlat, on="product_category_name", how="left")
products["category_en"] = products["product_category_name_english"].fillna(
    products["product_category_name"]
)
for c in ["product_name_lenght", "product_description_lenght",
          "product_photos_qty", "product_weight_g",
          "product_length_cm", "product_height_cm", "product_width_cm"]:
    products[c] = pd.to_numeric(products[c], errors="coerce")

# Fill median for physical measurements
for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products[c] = products[c].fillna(products[c].median())

products_slim = products[["product_id", "category_en", "product_weight_g"]].copy()
print(f"✅ Products cleaned  →  {products_slim.shape[0]:,} rows")


# ── 4g. SELLERS ──────────────────────────────────────────────
sellers["seller_city"]  = sellers["seller_city"].str.strip().str.title()
sellers["seller_state"] = sellers["seller_state"].str.strip().str.upper()
sellers.drop_duplicates(subset=["seller_id"], inplace=True)
print(f"✅ Sellers cleaned  →  {sellers.shape[0]:,} rows")


# ── CELL 5 · Merge into master dataset ────────────────────────
items_agg = (
    items.groupby("order_id")
    .agg(
        n_items           = ("order_item_id",    "max"),
        total_price       = ("price",            "sum"),
        total_freight     = ("freight_value",    "sum"),
        total_order_value = ("total_item_value", "sum"),
        product_id        = ("product_id",       "first"),
        seller_id         = ("seller_id",        "first"),
    )
    .reset_index()
)

df = (
    orders
    .merge(customers[["customer_id", "customer_city", "customer_state"]],
           on="customer_id", how="left")
    .merge(payments_agg,  on="order_id", how="left")
    .merge(reviews_slim,  on="order_id", how="left")
    .merge(items_agg,     on="order_id", how="left")
    .merge(products_slim, on="product_id", how="left")
    .merge(sellers[["seller_id", "seller_city", "seller_state"]],
           on="seller_id", how="left")
)

df.drop_duplicates(subset=["order_id"], inplace=True)
df.reset_index(drop=True, inplace=True)

# ✅ FIX 3: fill remaining nulls in category_en
df["category_en"] = df["category_en"].fillna("unknown")

print(f"\n✅ Master dataset  →  {df.shape[0]:,} rows × {df.shape[1]} columns")
print(df.dtypes.to_string())


# ── CELL 6 · Save processed dataset ──────────────────────────
OUTPUT_PATH = "/content/final_cleaned_data.csv"   # ✅ correct filename for assignment
df.to_csv(OUTPUT_PATH, index=False)
print(f"💾 Saved → {OUTPUT_PATH}")

from google.colab import files
files.download(OUTPUT_PATH)


# ══════════════════════════════════════════════════════════════
# ── CELL 7 · FRONTEND — Dataset preview & basic stats ─────────
# ══════════════════════════════════════════════════════════════

def html_block(title, body):
    return f"""
    <div style="font-family:Arial,sans-serif;margin:16px 0;
                border:1px solid #e0e0e0;border-radius:8px;overflow:hidden">
      <div style="background:#1a73e8;color:white;padding:10px 16px;
                  font-weight:bold;font-size:15px">{title}</div>
      <div style="padding:12px 16px;background:#fff">{body}</div>
    </div>"""

# ── Stat cards ────────────────────────────────────────────────
null_pct  = (df.isnull().sum().sum() / df.size * 100)
delivered = (df["order_status"] == "delivered").sum()

cards_html = "".join([
    f"""<div style="display:inline-block;background:#f1f3f4;border-radius:8px;
                    padding:14px 22px;margin:6px;text-align:center;min-width:140px">
          <div style="font-size:26px;font-weight:bold;color:#1a73e8">{v}</div>
          <div style="font-size:12px;color:#555;margin-top:4px">{k}</div>
        </div>"""
    for k, v in {
        "Total Orders"       : f"{df.shape[0]:,}",
        "Columns"            : df.shape[1],
        "Delivered Orders"   : f"{delivered:,}",
        "Avg Review Score"   : f"{df['review_score'].mean():.2f} ⭐",
        "Avg Order Value R$" : f"{df['total_order_value'].mean():.2f}",
        "Null Cells %"       : f"{null_pct:.1f}%",
    }.items()
])

display(HTML(html_block("📊 Dataset Overview", cards_html)))

# ── Table preview ─────────────────────────────────────────────
PREVIEW_COLS = [
    "order_id", "order_status", "order_purchase_timestamp",
    "customer_state", "n_items", "total_order_value",
    "payment_types", "review_score", "days_to_delivery",
    "delivery_delay_days", "category_en"
]
preview = df[PREVIEW_COLS].head(10).copy()
preview["order_purchase_timestamp"] = preview["order_purchase_timestamp"].dt.strftime("%Y-%m-%d")
preview["order_id"] = preview["order_id"].str[:12] + "…"

styled = (
    preview.style
    .set_table_styles([
        {"selector": "thead th",
         "props": [("background","#1a73e8"),("color","white"),
                   ("padding","8px 12px"),("font-size","13px")]},
        {"selector": "tbody tr:nth-child(even)",
         "props": [("background","#f8f9fa")]},
        {"selector": "td",
         "props": [("padding","7px 12px"),("font-size","13px")]},
    ])
    .format({
        "total_order_value"  : "R$ {:.2f}",
        "review_score"       : "{:.0f}",
        "n_items"            : "{:.0f}",
        "days_to_delivery"   : "{:.0f}",
        "delivery_delay_days": "{:.0f}",
    }, na_rep="—")
    .hide(axis="index")
)

display(HTML(html_block("🔍 Dataset Preview (first 10 rows)", styled.to_html())))

# ── Column-level null report ──────────────────────────────────
null_report = (
    df.isnull().sum()
    .rename("null_count")
    .to_frame()
    .assign(null_pct=lambda x: (x["null_count"] / len(df) * 100).round(1))
    .query("null_count > 0")
    .sort_values("null_pct", ascending=False)
)

if null_report.empty:
    null_html = "<p style='color:green;font-weight:bold'>🎉 No nulls remaining!</p>"
else:
    null_html = null_report.to_html(border=0, float_format="{:.1f}".format)

display(HTML(html_block("🔎 Remaining Nulls per Column", null_html)))

# ── Status breakdown ──────────────────────────────────────────
status_counts = df["order_status"].value_counts().reset_index()
status_counts.columns = ["Status", "Count"]
status_counts["Share %"] = (status_counts["Count"] / len(df) * 100).round(1)
display(HTML(html_block(
    "📦 Order Status Breakdown",
    status_counts.to_html(index=False, border=0)
)))

# ── Numeric summary ───────────────────────────────────────────
num_cols = ["total_order_value", "total_freight", "n_items",
            "review_score", "days_to_delivery", "delivery_delay_days"]
summary = df[num_cols].describe().round(2)
display(HTML(html_block("📈 Numeric Summary", summary.to_html(border=0))))

print("\n🎉 Pipeline complete! final_cleaned_data.csv is ready for your teammates.")


✅ Libraries ready
Upload your archive.zip ↓


Saving archive.zip to archive (2).zip

📂 Extracted 9 files:
   olist_customers_dataset.csv                                 9.0 MB
   olist_geolocation_dataset.csv                              61.3 MB
   olist_order_items_dataset.csv                              15.4 MB
   olist_order_payments_dataset.csv                            5.8 MB
   olist_order_reviews_dataset.csv                            14.5 MB
   olist_orders_dataset.csv                                   17.7 MB
   olist_products_dataset.csv                                  2.4 MB
   olist_sellers_dataset.csv                                   0.2 MB
   product_category_name_translation.csv                       0.0 MB
✅ All CSVs loaded
   orders         99,441 rows × 8 cols
   customers      99,441 rows × 5 cols
   items         112,650 rows × 7 cols
   payments      103,886 rows × 5 cols
   reviews        99,224 rows × 7 cols
   products       32,951 rows × 9 cols
   sellers         3,095 rows × 4 cols
✅ Orders cleaned  →

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

order_id,order_status,order_purchase_timestamp,customer_state,n_items,total_order_value,payment_types,review_score,days_to_delivery,delivery_delay_days,category_en
e481f51cbdc5…,delivered,2017-10-02,SP,1,R$ 38.71,credit_card|voucher,4,8,-8,housewares
53cdb2fc8bc7…,delivered,2018-07-24,BA,1,R$ 141.46,boleto,4,13,-6,perfumery
47770eb9100c…,delivered,2018-08-08,GO,1,R$ 179.12,credit_card,5,9,-18,auto
949d5b44dbf5…,delivered,2017-11-18,RN,1,R$ 72.20,credit_card,5,13,-13,pet_shop
ad21c59c0840…,delivered,2018-02-13,SP,1,R$ 28.62,credit_card,5,2,-10,stationery
a4591c265e18…,delivered,2017-07-09,PR,1,R$ 175.26,credit_card,4,16,-6,auto
136cce7faa42…,invoiced,2017-04-11,RS,1,R$ 65.95,credit_card,2,—,—,unknown
6514b8ad8028…,delivered,2017-05-16,RJ,1,R$ 75.16,credit_card,5,9,-12,auto
76c6e8662893…,delivered,2017-01-23,RS,1,R$ 35.95,boleto,1,9,-32,furniture_decor
e69bfb5eb88e…,delivered,2017-07-29,SP,1,R$ 169.76,voucher|credit_card,5,18,-7,office_furniture


,null_count,null_pct
order_delivered_customer_date,2351,2.4
delivery_delay_days,2351,2.4
days_to_delivery,2351,2.4
order_delivered_carrier_date,1169,1.2
review_comment_message,752,0.8
review_score,752,0.8
order_approved_at,155,0.2
n_items,167,0.2
total_order_value,167,0.2
total_freight,167,0.2


Status,Count,Share %
delivered,96478,97.6
shipped,1107,1.1
canceled,625,0.6
invoiced,314,0.3
processing,301,0.3
approved,2,0.0


,total_order_value,total_freight,n_items,review_score,days_to_delivery,delivery_delay_days
count,98660.00,98660.00,98660.00,98075.00,96476.00,96476.00
mean,160.57,22.82,1.14,4.10,12.09,-11.88
std,220.45,21.65,0.54,1.33,9.55,10.18
min,9.59,0.00,1.00,1.00,0.00,-147.00
25%,61.98,13.85,1.00,4.00,6.00,-17.00
50%,105.29,17.17,1.00,5.00,10.00,-12.00
75%,176.86,24.04,1.00,5.00,15.00,-7.00
max,13664.08,1794.96,21.00,5.00,209.00,188.00



🎉 Pipeline complete! final_cleaned_data.csv is ready for your teammates.
